<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Two paper findings reviewed:")
print("1. The CTR Cliff")
print("2. The Freshness Multiplier")
print("\nThe methodology questions are explained in the markdown cell above.")

Two paper findings reviewed:
1. The CTR Cliff
2. The Freshness Multiplier

The methodology questions are explained in the markdown cell above.


## 1. Two paper findings and my methodology questions

The FlyRank research paper was written for a broad audience and clearly separates its direct findings, myth tests, machine-learning analysis, and action playbooks. My purpose here is not to grade the paper. I am using two of its findings to practise asking respectful and concrete methodology questions.

### Finding 1: The CTR Cliff

The paper reports a measured relationship between average search position and click-through rate. The practical interpretation is that click-through performance changes substantially as pages move into weaker search positions.

My methodology question would be:

> How were pages with missing or zero average-position values treated, and were CTR comparisons made within sufficiently similar position groups, content types, clients, and impression levels?

I would ask this because average position and CTR are closely connected by definition. A page ranking near the top normally has a different click opportunity from a page ranking much lower. Large clients or high-impression pages could also dominate a portfolio-wide average.

A useful validation would report:

- The number of rows in each position group
- How `avg_position = 0` or missing position was handled
- Median as well as mean CTR
- Results by client or content type
- Whether the pattern remains after controlling for impression volume

This would help clarify whether the measured CTR cliff is broad across clients or concentrated in particular segments.

### Finding 2: The Freshness Multiplier

The paper reports a measured association between recently updated content and stronger search performance.

My methodology question would be:

> Does the comparison show that refreshing content caused stronger performance, or could pages have been selected for updating because they were already valuable, visible, or expected to improve?

I would ask this because updated and non-updated pages may differ before the refresh occurs. Teams may prioritise pages that already have high impressions, commercial value, strong rankings, or seasonal relevance.

A useful validation would compare:

- Performance before and after the update
- Updated pages against similar non-updated pages
- The same page across time
- Results across multiple clients
- The number of pages and observation windows in each group

A time-aware or matched comparison would support a stronger claim than a simple cross-sectional association.

My questions do not imply that the findings are wrong. They identify the additional evidence I would want before interpreting a measured association as a causal effect.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-09 — SECTION 2
# Week 5 model under random-row and grouped-client splits
# ============================================================

from pathlib import Path
import os
import json
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_SEED = 42

# ------------------------------------------------------------
# 1. Locate the repository and starter data
# ------------------------------------------------------------

REPO_NAME = "flyrank-ml-internship-sunaina"

REPO_URL = (
    "https://github.com/"
    "sunainakhatwani12/"
    "flyrank-ml-internship-sunaina.git"
)

possible_roots = [
    Path.cwd(),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

repo_root = None

for root in possible_roots:
    candidate = (
        root
        / "data"
        / "raw"
        / "content_refresh_anonymized.csv"
    )

    if candidate.exists():
        repo_root = root
        break

if repo_root is None:
    clone_path = Path("/content") / REPO_NAME

    if clone_path.exists():
        shutil.rmtree(clone_path)

    print("Repository files were not found.")
    print("Cloning the public repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, str(clone_path)],
        check=True,
    )

    repo_root = clone_path

os.chdir(repo_root)

data_path = (
    repo_root
    / "data"
    / "raw"
    / "content_refresh_anonymized.csv"
)

if not data_path.exists():
    raise FileNotFoundError(
        f"Starter data was not found at:\n{data_path}"
    )

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"Data path: {data_path}")
print(f"Output directory: {output_dir}")

# ------------------------------------------------------------
# 2. Load and verify data
# ------------------------------------------------------------

df = pd.read_csv(data_path)

required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "days_since_last_update",
    "impressions_90d",
]

missing_required = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}"
    )

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# FlyRank data rule:
# avg_position = 0 means position data is unavailable.
if "avg_position" in df.columns:
    df["avg_position_missing"] = (
        df["avg_position"]
        .fillna(0)
        .eq(0)
        .astype(int)
    )

    df.loc[
        df["avg_position"].eq(0),
        "avg_position",
    ] = np.nan

print("\nDATA SUMMARY")
print("-" * 70)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Clients: {df['client_id'].nunique():,}")
print(f"Content items: {df['content_id'].nunique():,}")
print(
    "Observed decline base rate: "
    f"{df['is_declining_label'].mean():.2%}"
)

# ------------------------------------------------------------
# 3. Define candidate features
# ------------------------------------------------------------

candidate_numeric_features = [
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "avg_position_missing",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc",
]

candidate_categorical_features = [
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [
    column
    for column in candidate_numeric_features
    if column in df.columns
]

categorical_features = [
    column
    for column in candidate_categorical_features
    if column in df.columns
]

feature_columns = numeric_features + categorical_features

if len(feature_columns) == 0:
    raise ValueError("No permitted model features were found.")

print("\nMODEL FEATURES")
print("-" * 70)
print(f"Numeric features ({len(numeric_features)}):")
print(numeric_features)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

# ------------------------------------------------------------
# 4. Shared preprocessing and model builder
# ------------------------------------------------------------

def build_model():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features,
            ),
        ],
        remainder="drop",
    )

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

# ------------------------------------------------------------
# 5. Ranking metrics
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k):
    results = pd.DataFrame(
        {
            "actual": np.asarray(y_true),
            "score": np.asarray(scores),
        }
    )

    results = results.sort_values(
        "score",
        ascending=False,
        kind="mergesort",
    )

    actual_k = min(k, len(results))

    if actual_k == 0:
        return np.nan

    return float(
        results.head(actual_k)["actual"].mean()
    )


def safe_roc_auc(y_true, scores):
    if pd.Series(y_true).nunique() < 2:
        return np.nan

    return float(
        roc_auc_score(y_true, scores)
    )


def evaluate_split(
    split_name,
    X_train,
    X_test,
    y_train,
    y_test,
    train_clients,
    test_clients,
):
    model = build_model()

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    overlap = set(train_clients).intersection(
        set(test_clients)
    )

    metrics = {
        "validation_design": split_name,
        "training_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "training_clients": int(
            pd.Series(train_clients).nunique()
        ),
        "test_clients": int(
            pd.Series(test_clients).nunique()
        ),
        "client_overlap_count": int(len(overlap)),
        "test_base_rate": float(y_test.mean()),
        "precision_at_20": precision_at_k(
            y_test,
            probabilities,
            20,
        ),
        "precision_at_50": precision_at_k(
            y_test,
            probabilities,
            50,
        ),
        "precision_at_100": precision_at_k(
            y_test,
            probabilities,
            100,
        ),
        "average_precision": float(
            average_precision_score(
                y_test,
                probabilities,
            )
        ),
        "roc_auc": safe_roc_auc(
            y_test,
            probabilities,
        ),
        "accuracy_at_050": float(
            accuracy_score(
                y_test,
                predictions,
            )
        ),
        "precision_at_050": float(
            precision_score(
                y_test,
                predictions,
                zero_division=0,
            )
        ),
        "recall_at_050": float(
            recall_score(
                y_test,
                predictions,
                zero_division=0,
            )
        ),
    }

    return model, probabilities, predictions, metrics

# ------------------------------------------------------------
# 6. BEFORE — random row split
# ------------------------------------------------------------

row_indices = np.arange(len(df))

(
    random_train_index,
    random_test_index,
) = train_test_split(
    row_indices,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_random_train = X.iloc[random_train_index].copy()
X_random_test = X.iloc[random_test_index].copy()

y_random_train = y.iloc[random_train_index].copy()
y_random_test = y.iloc[random_test_index].copy()

random_train_clients = groups.iloc[random_train_index]
random_test_clients = groups.iloc[random_test_index]

(
    random_model,
    random_probabilities,
    random_predictions,
    random_metrics,
) = evaluate_split(
    split_name="Before: random row split",
    X_train=X_random_train,
    X_test=X_random_test,
    y_train=y_random_train,
    y_test=y_random_test,
    train_clients=random_train_clients,
    test_clients=random_test_clients,
)

# ------------------------------------------------------------
# 7. AFTER — grouped client split
# ------------------------------------------------------------

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

group_train_index, group_test_index = next(
    group_split.split(
        X,
        y,
        groups=groups,
    )
)

X_group_train = X.iloc[group_train_index].copy()
X_group_test = X.iloc[group_test_index].copy()

y_group_train = y.iloc[group_train_index].copy()
y_group_test = y.iloc[group_test_index].copy()

group_train_clients = groups.iloc[group_train_index]
group_test_clients = groups.iloc[group_test_index]

(
    grouped_model,
    grouped_probabilities,
    grouped_predictions,
    grouped_metrics,
) = evaluate_split(
    split_name="After: grouped client split",
    X_train=X_group_train,
    X_test=X_group_test,
    y_train=y_group_train,
    y_test=y_group_test,
    train_clients=group_train_clients,
    test_clients=group_test_clients,
)

# ------------------------------------------------------------
# 8. Before/after comparison
# ------------------------------------------------------------

comparison = pd.DataFrame(
    [
        random_metrics,
        grouped_metrics,
    ]
)

metric_columns = [
    "validation_design",
    "training_rows",
    "test_rows",
    "training_clients",
    "test_clients",
    "client_overlap_count",
    "test_base_rate",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100",
    "average_precision",
    "roc_auc",
]

print("\nBEFORE/AFTER VALIDATION COMPARISON")
print("-" * 70)

display(
    comparison[metric_columns].style.format(
        {
            "test_base_rate": "{:.2%}",
            "precision_at_20": "{:.2%}",
            "precision_at_50": "{:.2%}",
            "precision_at_100": "{:.2%}",
            "average_precision": "{:.4f}",
            "roc_auc": "{:.4f}",
        }
    )
)

random_overlap = random_metrics[
    "client_overlap_count"
]

grouped_overlap = grouped_metrics[
    "client_overlap_count"
]

print("\nSPLIT AUDIT")
print("-" * 70)
print(
    "Random-row client overlap: "
    f"{random_overlap}"
)
print(
    "Grouped-split client overlap: "
    f"{grouped_overlap}"
)

assert grouped_overlap == 0, (
    "The grouped split contains client leakage."
)

print("\nINTERPRETATION")
print("-" * 70)

print(
    "The random-row result measures performance "
    "when pages from previously seen clients may "
    "appear in the test set."
)

print(
    "The grouped result measures performance on "
    "held-out clients and is the more appropriate "
    "estimate for cross-client transfer."
)

if (
    grouped_metrics["precision_at_50"]
    < random_metrics["precision_at_50"]
):
    print(
        "Observed Precision@50 is lower under the "
        "grouped split. This suggests the random-row "
        "split gave a more optimistic estimate."
    )
elif (
    grouped_metrics["precision_at_50"]
    > random_metrics["precision_at_50"]
):
    print(
        "Observed Precision@50 is higher under the "
        "grouped split. This result is possible, but "
        "the grouped estimate should still be preferred "
        "because it prevents client overlap."
    )
else:
    print(
        "Observed Precision@50 is equal under both "
        "splits. The grouped design remains preferable "
        "because it prevents client overlap."
    )

# ------------------------------------------------------------
# 9. Save comparison receipt
# ------------------------------------------------------------

comparison_path = (
    output_dir
    / "w06_split_comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False,
)

print(
    "\nSplit comparison written to: "
    f"{comparison_path}"
)

Repository files were not found.
Cloning the public repository...
Repository root: /content/flyrank-ml-internship-sunaina
Data path: /content/flyrank-ml-internship-sunaina/data/raw/content_refresh_anonymized.csv
Output directory: /content/flyrank-ml-internship-sunaina/work/outputs

DATA SUMMARY
----------------------------------------------------------------------
Rows: 30,000
Columns: 46
Clients: 32
Content items: 30,000
Observed decline base rate: 54.21%

MODEL FEATURES
----------------------------------------------------------------------
Numeric features (23):
['days_since_last_update', 'content_age_days', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'avg_position_missing', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']

Categorical features (8):

,validation_design,training_rows,test_rows,training_clients,test_clients,client_overlap_count,test_base_rate,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc
0,Before: random row split,22500,7500,32,31,31,54.20%,90.00%,90.00%,91.00%,0.7620,0.7507
1,After: grouped client split,22885,7115,24,8,0,51.65%,50.00%,54.00%,49.00%,0.5848,0.6037



SPLIT AUDIT
----------------------------------------------------------------------
Random-row client overlap: 31
Grouped-split client overlap: 0

INTERPRETATION
----------------------------------------------------------------------
The random-row result measures performance when pages from previously seen clients may appear in the test set.
The grouped result measures performance on held-out clients and is the more appropriate estimate for cross-client transfer.
Observed Precision@50 is lower under the grouped split. This suggests the random-row split gave a more optimistic estimate.

Split comparison written to: /content/flyrank-ml-internship-sunaina/work/outputs/w06_split_comparison.csv


## 2. My model under an honest split

My Week 5 model ranks content pages for possible refresh review. The target is an observed decline label derived from `trend_direction`.

To audit the validation design, I compare two versions of the same Random Forest model:

1. **Before improvement: random row split**
2. **After improvement: grouped split by `client_id`**

The random row split can place pages from the same client in both training and test data. Pages from one client may share audience, content strategy, measurement setup, publishing behaviour, and domain-level search patterns. This can make test performance look stronger than performance on genuinely unseen clients.

The grouped split keeps every client entirely in either training or test data. It therefore measures how well the model transfers to held-out clients.

Both versions use:

- The same source dataset
- The same target
- The same allowed features
- The same preprocessing
- The same Random Forest settings
- The same ranking metrics
- Random seed 42

The main decision-support metrics are Precision@20, Precision@50, and Precision@100. I also report average precision and ROC-AUC as supporting measures.

The grouped result is the more honest estimate for claims about transfer to unseen clients. The random-row result is included only as a before/after validation comparison.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-09 — SECTION 3
# Explicit feature and validation leakage audit
# ============================================================

# ------------------------------------------------------------
# 1. Define prohibited inputs
# ------------------------------------------------------------

direct_label_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

identifier_columns = {
    "content_id",
    "client_id",
}

future_or_comparison_window_columns = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "users_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "users_prev_30d",
    "impressions_change_pct",
    "clicks_change_pct",
    "sessions_change_pct",
}

existing_action_columns = {
    "needs_refresh",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_flag",
    "action_flag",
    "action_label",
    "reason_code",
    "baseline_action_score",
}

forbidden_columns = (
    direct_label_columns
    | identifier_columns
    | future_or_comparison_window_columns
    | existing_action_columns
)

leaked_features = sorted(
    set(feature_columns).intersection(
        forbidden_columns
    )
)

# ------------------------------------------------------------
# 2. Name-based suspicious-column scan
# ------------------------------------------------------------

suspicious_terms = [
    "trend",
    "target",
    "label",
    "future",
    "next_",
    "prev_",
    "change",
    "growth",
    "declin",
    "refresh_flag",
    "action",
]

name_scan = []

for column in feature_columns:
    matched_terms = [
        term
        for term in suspicious_terms
        if term.lower() in column.lower()
    ]

    if matched_terms:
        name_scan.append(
            {
                "feature": column,
                "matched_terms": ", ".join(
                    matched_terms
                ),
            }
        )

name_scan_table = pd.DataFrame(
    name_scan,
    columns=[
        "feature",
        "matched_terms",
    ],
)

# ------------------------------------------------------------
# 3. Audit table
# ------------------------------------------------------------

audit_rows = [
    {
        "audit_item": "Direct label-derived inputs",
        "result": (
            "PASS"
            if len(
                set(feature_columns)
                .intersection(
                    direct_label_columns
                )
            ) == 0
            else "FAIL"
        ),
        "details": str(
            sorted(
                set(feature_columns)
                .intersection(
                    direct_label_columns
                )
            )
        ),
    },
    {
        "audit_item": "Identifiers used as features",
        "result": (
            "PASS"
            if len(
                set(feature_columns)
                .intersection(
                    identifier_columns
                )
            ) == 0
            else "FAIL"
        ),
        "details": str(
            sorted(
                set(feature_columns)
                .intersection(
                    identifier_columns
                )
            )
        ),
    },
    {
        "audit_item": "Comparison/future windows",
        "result": (
            "PASS"
            if len(
                set(feature_columns)
                .intersection(
                    future_or_comparison_window_columns
                )
            ) == 0
            else "FAIL"
        ),
        "details": str(
            sorted(
                set(feature_columns)
                .intersection(
                    future_or_comparison_window_columns
                )
            )
        ),
    },
    {
        "audit_item": "Existing action flags",
        "result": (
            "PASS"
            if len(
                set(feature_columns)
                .intersection(
                    existing_action_columns
                )
            ) == 0
            else "FAIL"
        ),
        "details": str(
            sorted(
                set(feature_columns)
                .intersection(
                    existing_action_columns
                )
            )
        ),
    },
    {
        "audit_item": "Grouped client overlap",
        "result": (
            "PASS"
            if grouped_overlap == 0
            else "FAIL"
        ),
        "details": (
            f"{grouped_overlap} overlapping clients"
        ),
    },
    {
        "audit_item": "Random-row client overlap",
        "result": (
            "EXPECTED WARNING"
            if random_overlap > 0
            else "PASS"
        ),
        "details": (
            f"{random_overlap} overlapping clients"
        ),
    },
    {
        "audit_item": "avg_position zero handling",
        "result": (
            "PASS"
            if (
                "avg_position" not in df.columns
                or not df["avg_position"].eq(0).any()
            )
            else "FAIL"
        ),
        "details": (
            "Zero values were changed to missing "
            "and a missingness flag was created."
            if "avg_position" in df.columns
            else "avg_position was unavailable."
        ),
    },
]

leakage_audit = pd.DataFrame(audit_rows)

print("LEAKAGE AUDIT")
print("-" * 70)

display(leakage_audit)

print("\nFINAL MODEL FEATURE LIST")
print("-" * 70)
print(feature_columns)

print("\nFORBIDDEN FEATURES FOUND")
print("-" * 70)
print(leaked_features)

print("\nNAME-BASED SUSPICIOUS FEATURE SCAN")
print("-" * 70)

if len(name_scan_table) == 0:
    print("No feature names triggered the suspicious-term scan.")
else:
    display(name_scan_table)

# ------------------------------------------------------------
# 4. Assertions
# ------------------------------------------------------------

assert leaked_features == [], (
    f"Forbidden features detected: {leaked_features}"
)

assert "client_id" not in feature_columns
assert "content_id" not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert "is_declining_label" not in feature_columns
assert grouped_overlap == 0

if "avg_position" in df.columns:
    assert not df["avg_position"].eq(0).any(), (
        "avg_position still contains zero values."
    )

print("\nLEAKAGE AUDIT RESULT")
print("-" * 70)
print("Direct target leakage found: NO")
print("Identifiers used as predictive features: NO")
print("Existing action flags used: NO")
print("Comparison or future windows used: NO")
print("Grouped split client overlap: NONE")
print("All leakage assertions passed.")

# ------------------------------------------------------------
# 5. Save leakage receipt
# ------------------------------------------------------------

leakage_path = (
    output_dir
    / "w06_leakage_audit.csv"
)

leakage_audit.to_csv(
    leakage_path,
    index=False,
)

print(
    "\nLeakage audit written to: "
    f"{leakage_path}"
)

LEAKAGE AUDIT
----------------------------------------------------------------------


,audit_item,result,details
0,Direct label-derived inputs,PASS,[]
1,Identifiers used as features,PASS,[]
2,Comparison/future windows,PASS,[]
3,Existing action flags,PASS,[]
4,Grouped client overlap,PASS,0 overlapping clients
5,Random-row client overlap,EXPECTED WARNING,31 overlapping clients
6,avg_position zero handling,PASS,Zero values were changed to missing and a missingness flag was created.



FINAL MODEL FEATURE LIST
----------------------------------------------------------------------
['days_since_last_update', 'content_age_days', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'avg_position_missing', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

FORBIDDEN FEATURES FOUND
----------------------------------------------------------------------
[]

NAME-BASED SUSPICIOUS FEATURE SCAN
----------------------------------------------------------------------
No feature names triggered the suspicious-term scan.

LEAKAGE AUDIT RESULT
----------------------------------------------------------------------
Direct targe

## 3. Leakage audit

I audited the target, features, identifiers, windows, preprocessing, and validation design.

### Label source

The target `is_declining_label` is created from `trend_direction`.

This is an observed directional proxy. It does not directly measure whether a page needs a refresh, whether a refresh will work, or whether the page has business value.

### Features excluded from prediction

I exclude:

- `trend_direction`
- `trend_pct`
- `is_declining_label`
- Recent-window versus previous-window outcome fields
- Existing FlyRank optimisation or action flags
- `content_id`
- `client_id`

`client_id` is used only to form the grouped split. It is not passed to the model as a predictive feature.

### Position handling

The dataset rule says `avg_position = 0` represents missing position data. I replace zero with missing and create a missingness indicator instead of interpreting zero as the strongest possible search position.

### Remaining proxy risk

Even after direct leakage is removed, some permitted features may be strongly related to the observed decline label because they measure recent search behaviour. This is not necessarily direct leakage, but it limits interpretation.

The model learns associations in this data extract. It does not establish that any feature caused decline or that changing the feature will cause recovery.

### Validation leakage

The random-row split has client overlap and is therefore marked as the weaker “before” design. The grouped split has zero client overlap and is used for my final measured claim.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-09 — SECTION 4
# Failure examples and evidence-aligned claim receipt
# ============================================================

# ------------------------------------------------------------
# 1. Build grouped-test prediction table
# ------------------------------------------------------------

grouped_test_rows = (
    df.iloc[group_test_index]
    .copy()
    .reset_index(drop=True)
)

grouped_test_rows[
    "predicted_probability"
] = grouped_probabilities

grouped_test_rows[
    "predicted_label_at_050"
] = grouped_predictions

grouped_test_rows[
    "actual_label"
] = y_group_test.to_numpy()

grouped_test_rows[
    "error_type"
] = np.select(
    [
        (
            grouped_test_rows["actual_label"] == 0
        )
        & (
            grouped_test_rows[
                "predicted_label_at_050"
            ] == 1
        ),
        (
            grouped_test_rows["actual_label"] == 1
        )
        & (
            grouped_test_rows[
                "predicted_label_at_050"
            ] == 0
        ),
    ],
    [
        "False positive",
        "False negative",
    ],
    default="Correct",
)

grouped_test_rows[
    "wrong_prediction_confidence"
] = np.where(
    grouped_test_rows["error_type"]
    .eq("False positive"),
    grouped_test_rows[
        "predicted_probability"
    ],
    np.where(
        grouped_test_rows["error_type"]
        .eq("False negative"),
        1
        - grouped_test_rows[
            "predicted_probability"
        ],
        0,
    ),
)

# ------------------------------------------------------------
# 2. Error counts and confusion matrix
# ------------------------------------------------------------

confusion = confusion_matrix(
    grouped_test_rows["actual_label"],
    grouped_test_rows[
        "predicted_label_at_050"
    ],
)

confusion_table = pd.DataFrame(
    confusion,
    index=[
        "Actual not declining",
        "Actual declining",
    ],
    columns=[
        "Predicted not declining",
        "Predicted declining",
    ],
)

print("GROUPED-TEST CONFUSION MATRIX")
print("-" * 70)
display(confusion_table)

error_counts = (
    grouped_test_rows["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="n")
)

error_counts["share_of_test"] = (
    error_counts["n"]
    / len(grouped_test_rows)
)

print("\nERROR COUNTS")
print("-" * 70)

display(
    error_counts.style.format(
        {
            "share_of_test": "{:.2%}",
        }
    )
)

# ------------------------------------------------------------
# 3. Select real failure examples
# ------------------------------------------------------------

false_positives = (
    grouped_test_rows.loc[
        grouped_test_rows["error_type"]
        == "False positive"
    ]
    .sort_values(
        "wrong_prediction_confidence",
        ascending=False,
    )
)

false_negatives = (
    grouped_test_rows.loc[
        grouped_test_rows["error_type"]
        == "False negative"
    ]
    .sort_values(
        "wrong_prediction_confidence",
        ascending=False,
    )
)

selected_failures = pd.concat(
    [
        false_positives.head(2),
        false_negatives.head(2),
    ],
    ignore_index=True,
)

if len(selected_failures) < 3:
    all_errors = (
        grouped_test_rows.loc[
            grouped_test_rows["error_type"]
            != "Correct"
        ]
        .sort_values(
            "wrong_prediction_confidence",
            ascending=False,
        )
    )

    selected_failures = (
        all_errors.head(4).copy()
    )

# ------------------------------------------------------------
# 4. Add safe interpretation
# ------------------------------------------------------------

def failure_explanation(row):
    signals = []

    staleness = row.get(
        "days_since_last_update",
        np.nan,
    )

    impressions = row.get(
        "impressions_90d",
        np.nan,
    )

    position = row.get(
        "avg_position",
        np.nan,
    )

    ctr = row.get(
        "ctr",
        np.nan,
    )

    if pd.notna(staleness):
        if staleness >= 180:
            signals.append(
                "measured staleness is high"
            )
        else:
            signals.append(
                "measured staleness is relatively low"
            )

    if pd.notna(impressions):
        if impressions >= 300:
            signals.append(
                "the page has meaningful measured impressions"
            )
        else:
            signals.append(
                "the page has limited measured impressions"
            )

    if pd.notna(position):
        if position <= 10:
            signals.append(
                "average search position is relatively strong"
            )
        elif position > 50:
            signals.append(
                "average search position is relatively weak"
            )

    if pd.notna(ctr):
        if ctr < 0.5:
            signals.append(
                "measured CTR is low"
            )
        elif ctr >= 3:
            signals.append(
                "measured CTR is relatively healthy"
            )

    signal_text = "; ".join(signals)

    return (
        f"{signal_text}. These measured signals may "
        "point in different directions. The model cannot "
        "observe topic seasonality, editorial accuracy, "
        "technical issues, business priority, or recent "
        "work outside the data extract."
    )


selected_failures[
    "safe_interpretation"
] = selected_failures.apply(
    failure_explanation,
    axis=1,
)

failure_columns = [
    column
    for column in [
        "error_type",
        "actual_label",
        "predicted_label_at_050",
        "predicted_probability",
        "days_since_last_update",
        "content_age_days",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "sessions_90d",
        "content_type",
        "main_intent",
        "safe_interpretation",
    ]
    if column in selected_failures.columns
]

print("\nREAL FAILURE EXAMPLES")
print("-" * 70)

if len(selected_failures) == 0:
    print(
        "No threshold-based errors were available "
        "for inspection."
    )
else:
    display(
        selected_failures[
            failure_columns
        ]
    )

# ------------------------------------------------------------
# 5. Compare grouped model with Week 4 baseline
# ------------------------------------------------------------

STALE_THRESHOLD_DAYS = 180
VISIBILITY_THRESHOLD_IMPRESSIONS = 300

baseline_train = X_group_train.copy()
baseline_test = X_group_test.copy()

staleness_cap = max(
    STALE_THRESHOLD_DAYS + 1,
    float(
        baseline_train[
            "days_since_last_update"
        ].quantile(0.95)
    ),
)

impression_cap = max(
    VISIBILITY_THRESHOLD_IMPRESSIONS + 1,
    float(
        baseline_train[
            "impressions_90d"
        ].quantile(0.95)
    ),
)


def make_baseline_scores(data):
    eligible = (
        (
            data["days_since_last_update"]
            >= STALE_THRESHOLD_DAYS
        )
        & (
            data["impressions_90d"]
            >= VISIBILITY_THRESHOLD_IMPRESSIONS
        )
    )

    staleness_score = (
        data["days_since_last_update"]
        .clip(
            lower=STALE_THRESHOLD_DAYS,
            upper=staleness_cap,
        )
        .sub(STALE_THRESHOLD_DAYS)
        .div(
            staleness_cap
            - STALE_THRESHOLD_DAYS
        )
        .mul(60)
    )

    log_floor = np.log1p(
        VISIBILITY_THRESHOLD_IMPRESSIONS
    )

    log_cap = np.log1p(
        impression_cap
    )

    visibility_score = (
        np.log1p(
            data["impressions_90d"]
            .clip(
                lower=(
                    VISIBILITY_THRESHOLD_IMPRESSIONS
                ),
                upper=impression_cap,
            )
        )
        .sub(log_floor)
        .div(log_cap - log_floor)
        .mul(40)
    )

    score = (
        staleness_score
        + visibility_score
    )

    score = score.where(
        eligible,
        0,
    )

    return score.clip(0, 100)


baseline_scores = make_baseline_scores(
    X_group_test
)

baseline_vs_model = pd.DataFrame(
    [
        {
            "method": "Week 4 rule baseline",
            "precision_at_20": precision_at_k(
                y_group_test,
                baseline_scores,
                20,
            ),
            "precision_at_50": precision_at_k(
                y_group_test,
                baseline_scores,
                50,
            ),
            "precision_at_100": precision_at_k(
                y_group_test,
                baseline_scores,
                100,
            ),
        },
        {
            "method": "Random Forest",
            "precision_at_20": precision_at_k(
                y_group_test,
                grouped_probabilities,
                20,
            ),
            "precision_at_50": precision_at_k(
                y_group_test,
                grouped_probabilities,
                50,
            ),
            "precision_at_100": precision_at_k(
                y_group_test,
                grouped_probabilities,
                100,
            ),
        },
    ]
)

print("\nBASELINE VS MODEL ON THE SAME GROUPED TEST ROWS")
print("-" * 70)

display(
    baseline_vs_model.style.format(
        {
            "precision_at_20": "{:.2%}",
            "precision_at_50": "{:.2%}",
            "precision_at_100": "{:.2%}",
        }
    )
)

# ------------------------------------------------------------
# 6. Generate evidence-aligned claim
# ------------------------------------------------------------

baseline_p50 = float(
    baseline_vs_model.loc[
        baseline_vs_model["method"]
        == "Week 4 rule baseline",
        "precision_at_50",
    ].iloc[0]
)

model_p50 = float(
    baseline_vs_model.loc[
        baseline_vs_model["method"]
        == "Random Forest",
        "precision_at_50",
    ].iloc[0]
)

if model_p50 > baseline_p50:
    measured_comparison_claim = (
        "On this grouped held-out client split, "
        "the Random Forest produced higher measured "
        f"Precision@50 ({model_p50:.2%}) than the "
        f"Week 4 rule baseline ({baseline_p50:.2%}). "
        "This observed difference applies to this "
        "dataset and validation setup and does not "
        "establish universal superiority."
    )
elif model_p50 < baseline_p50:
    measured_comparison_claim = (
        "On this grouped held-out client split, "
        "the Week 4 rule baseline produced higher "
        f"measured Precision@50 ({baseline_p50:.2%}) "
        f"than the Random Forest ({model_p50:.2%}). "
        "The learned model therefore did not improve "
        "this metric in this validation setup."
    )
else:
    measured_comparison_claim = (
        "On this grouped held-out client split, "
        "the Random Forest and Week 4 baseline "
        f"both produced measured Precision@50 of "
        f"{model_p50:.2%}. The learned model did "
        "not improve this metric in this setup."
    )

print("\nEVIDENCE-ALIGNED RESULT CLAIM")
print("-" * 70)
print(measured_comparison_claim)

# ------------------------------------------------------------
# 7. Save failure and claim receipts
# ------------------------------------------------------------

failure_path = (
    output_dir
    / "w06_failure_examples.csv"
)

baseline_comparison_path = (
    output_dir
    / "w06_baseline_model_comparison.csv"
)

claim_path = (
    output_dir
    / "w06_safe_claim.json"
)

selected_failures[
    failure_columns
].to_csv(
    failure_path,
    index=False,
)

baseline_vs_model.to_csv(
    baseline_comparison_path,
    index=False,
)

with open(
    claim_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "lane": (
                "Refresh / Content Opportunity Scoring"
            ),
            "validation_design": (
                "Grouped client split"
            ),
            "safe_claim": (
                measured_comparison_claim
            ),
            "limitations": [
                (
                    "The decline label is a proxy "
                    "derived from trend_direction."
                ),
                (
                    "The model measures association, "
                    "not causal refresh impact."
                ),
                (
                    "The model cannot observe all "
                    "editorial and business context."
                ),
            ],
        },
        file,
        indent=2,
    )

print("\nOUTPUT FILES")
print("-" * 70)
print(f"Failure examples: {failure_path}")
print(
    "Baseline/model comparison: "
    f"{baseline_comparison_path}"
)
print(f"Safe claim receipt: {claim_path}")

GROUPED-TEST CONFUSION MATRIX
----------------------------------------------------------------------


,Predicted not declining,Predicted declining
Actual not declining,1657,1783
Actual declining,1201,2474



ERROR COUNTS
----------------------------------------------------------------------


,error_type,n,share_of_test
0,Correct,4131,58.06%
1,False positive,1783,25.06%
2,False negative,1201,16.88%



REAL FAILURE EXAMPLES
----------------------------------------------------------------------


,error_type,actual_label,predicted_label_at_050,predicted_probability,days_since_last_update,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,content_type,main_intent,safe_interpretation
0,False positive,0,1,0.8185,103,238,1191,0,0.0000,23.1000,5,keyword article,informational,measured staleness is relatively low; the page has meaningful measured impressions; measured CTR is low. These measured signals may poin...
1,False positive,0,1,0.8173,104,275,335,0,0.0000,31.3000,4,keyword article,informational,measured staleness is relatively low; the page has meaningful measured impressions; measured CTR is low. These measured signals may poin...
2,False negative,1,0,0.0822,92,238,1,0,0.0000,NaN,1,keyword article,transactional,measured staleness is relatively low; the page has limited measured impressions; measured CTR is low. These measured signals may point i...
3,False negative,1,0,0.1267,20,358,2,0,0.0000,50.0000,2,keyword article,informational,measured staleness is relatively low; the page has limited measured impressions; measured CTR is low. These measured signals may point i...



BASELINE VS MODEL ON THE SAME GROUPED TEST ROWS
----------------------------------------------------------------------


,method,precision_at_20,precision_at_50,precision_at_100
0,Week 4 rule baseline,75.00%,70.00%,65.00%
1,Random Forest,50.00%,54.00%,49.00%



EVIDENCE-ALIGNED RESULT CLAIM
----------------------------------------------------------------------
On this grouped held-out client split, the Week 4 rule baseline produced higher measured Precision@50 (70.00%) than the Random Forest (54.00%). The learned model therefore did not improve this metric in this validation setup.

OUTPUT FILES
----------------------------------------------------------------------
Failure examples: /content/flyrank-ml-internship-sunaina/work/outputs/w06_failure_examples.csv
Baseline/model comparison: /content/flyrank-ml-internship-sunaina/work/outputs/w06_baseline_model_comparison.csv
Safe claim receipt: /content/flyrank-ml-internship-sunaina/work/outputs/w06_safe_claim.json


## 4. Claim rewrite and failure examples

### Claim 1

**Too strong:**

> The model identifies pages that need to be refreshed.

**Evidence-aligned rewrite:**

> On the held-out clients in this dataset, the model produced a ranked list in which the top positions contained a measured concentration of pages with the observed decline label. The ranking can support human review for possible refresh opportunities.

The revised claim does not state that every recommendation needs a refresh or that refreshing will cause improvement.

### Claim 2

**Too strong:**

> Random Forest is better than the baseline.

**Evidence-aligned rewrite:**

> Under the grouped client split used in this notebook, the Random Forest's measured Precision@K can be compared with the Week 4 baseline on the same held-out rows. Any observed difference applies to this dataset, split, target proxy, and evaluation setup.

This rewrite avoids claiming universal superiority.

### Claim 3

**Too strong:**

> Old content declines because it has not been updated.

**Evidence-aligned rewrite:**

> Content age and days since update may show a directional association with the observed decline label. The available data does not establish that staleness caused the decline.

### Claim 4

**Too strong:**

> A high predicted probability means the page will improve after a refresh.

**Evidence-aligned rewrite:**

> A high predicted probability means the page resembles examples associated with the observed decline label in the training data. It does not estimate the causal benefit of refreshing the page.

### Claim 5

**Too strong:**

> The model will work for every FlyRank client.

**Evidence-aligned rewrite:**

> The grouped test provides a more honest measured estimate on held-out clients from this release. Further validation across later time periods and additional clients would be needed before making a broad production claim.

### Failure-example interpretation

False positives are pages that the model ranks as likely declining even though their observed label is not declining. False negatives are declining-labelled pages that receive a lower model probability.

These errors may occur because the model cannot observe:

- Topic seasonality
- Editorial quality
- Search-intent changes
- Technical SEO issues
- Business or conversion value
- Recently completed work not represented in the extract
- Whether a page is intentionally evergreen
- Whether consolidation, redirection, or deletion is the better action

These limitations support using the ranking as decision support rather than automatic action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# ============================================================
# ML-09 — SECTION 5
# Final assignment verification
# ============================================================

required_output_paths = [
    comparison_path,
    leakage_path,
    failure_path,
    baseline_comparison_path,
    claim_path,
]

for path in required_output_paths:
    assert path.exists(), (
        f"Required output was not written: {path}"
    )

assert grouped_overlap == 0
assert leaked_features == []
assert "client_id" not in feature_columns
assert "content_id" not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert "is_declining_label" not in feature_columns

assert comparison[
    "precision_at_20"
].between(0, 1).all()

assert comparison[
    "precision_at_50"
].between(0, 1).all()

assert comparison[
    "precision_at_100"
].between(0, 1).all()

assert baseline_vs_model[
    "precision_at_20"
].between(0, 1).all()

assert baseline_vs_model[
    "precision_at_50"
].between(0, 1).all()

assert baseline_vs_model[
    "precision_at_100"
].between(0, 1).all()

print("FINAL WEEK 6 SELF-CHECK")
print("-" * 70)
print("Two paper findings reviewed: YES")
print("Methodology questions written: YES")
print("Random-row split evaluated: YES")
print("Grouped-client split evaluated: YES")
print("Before/after comparison included: YES")
print("Grouped split client overlap: NONE")
print("Leakage audit completed: YES")
print("Direct label leakage found: NO")
print("Identifiers used as model features: NO")
print("Existing action flags used: NO")
print("Real failure examples inspected: YES")
print("Baseline and model compared fairly: YES")
print("Evidence-aligned claim generated: YES")
print("Public-safe language used: YES")
print("Required output files written: YES")
print("All verification assertions passed.")

print("\nFINAL MEASURED CLAIM")
print("-" * 70)
print(measured_comparison_claim)

print("\nNOTE")
print("-" * 70)
print(
    "The ranking is decision support. It identifies "
    "measured patterns associated with the observed "
    "decline label and does not prove that refreshing "
    "a page will cause performance to improve."
)

FINAL WEEK 6 SELF-CHECK
----------------------------------------------------------------------
Two paper findings reviewed: YES
Methodology questions written: YES
Random-row split evaluated: YES
Grouped-client split evaluated: YES
Before/after comparison included: YES
Grouped split client overlap: NONE
Leakage audit completed: YES
Direct label leakage found: NO
Identifiers used as model features: NO
Existing action flags used: NO
Real failure examples inspected: YES
Baseline and model compared fairly: YES
Evidence-aligned claim generated: YES
Public-safe language used: YES
Required output files written: YES
All verification assertions passed.

FINAL MEASURED CLAIM
----------------------------------------------------------------------
On this grouped held-out client split, the Week 4 rule baseline produced higher measured Precision@50 (70.00%) than the Random Forest (54.00%). The learned model therefore did not improve this metric in this validation setup.

NOTE
------------------------